# Imports

In [ ]:
import os
import sys
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

from uavloc.datasets import dataset_ref
from uavloc.features.segmentation import load_segmenter
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      bilinear_argmax, evaluate, load_val_sets, results_table, run_model)
from uavloc.segmentation.latency import model_latency, standard_tile
from uavloc.segmentation.losses import boundary_ce, ce_dice_loss
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
EXP = "016-boundary-loss"
NOTE = ""
LOG_RUN = True

LAMBDA_BOUNDARY = 0.0
BAND_PX = 4
STEPS = 400
LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
SAT_CROP_PX = 512
SAT_BATCH = 4
RARE_CROP_P = 0.3
GAP_PX = 64
RECALIBRATE_BN = True
BN_BATCHES = 60
BN_SEED = 123
SEED = 42
DEVICE = "mps"
EVAL_DEVICE = "cpu"
TRAIN_THREADS = 4
LOG_EVERY_STEPS = 25
DATASET = "ua-village-annotated"
INIT_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
LATENCY_TILE_DATE = "2024-06-06"
LATENCY_TILE_TOP_LEFT_PX = (1024, 1024)
LATENCY_BUDGET_MS = 125.0

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TRAIN_THREADS)
DERIVED = REPO_ROOT / "data" / "derived"
ARTIFACTS = experiment_dir(EXP) / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
sat = vs.load_satellite(DERIVED / DATASET, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
val_sets = load_val_sets(DATASET)
{name: len(images) for name, images in val_sets.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / INIT_CHECKPOINT, DEVICE, architecture="lraspp")
n_params_initial = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=max(STEPS, 1), pct_start=WARMUP_FRAC)
n_params_initial

In [ ]:
def training_step(x: torch.Tensor, y: torch.Tensor) -> tuple[float, float]:
    """One optimiser step on CE + Dice + LAMBDA_BOUNDARY * band CE; returns (CE + Dice, band CE)."""
    logits = model(x.to(DEVICE))
    y = y.to(DEVICE)
    base, band = ce_dice_loss(logits, y), boundary_ce(logits, y, BAND_PX)
    loss = base + LAMBDA_BOUNDARY * band
    opt.zero_grad()
    loss.backward()
    opt.step()
    sched.step()
    return base.item(), band.item()

# Run

In [ ]:
rng = np.random.default_rng(SEED)
history, window, t0 = [], [], time.time()
model.train()
for step in range(1, STEPS + 1):
    window.append(training_step(*vs.batch_of(sampler, rng, SAT_BATCH)))
    if step % LOG_EVERY_STEPS == 0 or step == STEPS:
        base, band = np.mean(window, axis=0)
        history.append({"step": step, "ce_dice": float(base), "band_ce": float(band), "minutes": (time.time() - t0) / 60})
        print(f"step {step:4d}  ce+dice {base:.3f}  band ce {band:.3f}  [{history[-1]['minutes']:.1f} min]", flush=True)
        window = []
train_minutes = (time.time() - t0) / 60

In [ ]:
if RECALIBRATE_BN:
    model = vs.domain_bn(model, sampler, SAT_BATCH, BN_BATCHES, BN_SEED, torch.device(DEVICE))
model.eval()
checkpoint = ARTIFACTS / f"model-lambda{LAMBDA_BOUNDARY:g}-seed{SEED}-steps{STEPS}.pt" if STEPS else None
if checkpoint:
    torch.save(model.state_dict(), checkpoint)
checkpoint

In [ ]:
cases = run_model(model.to(EVAL_DEVICE), val_sets, EVAL_DEVICE)
results = evaluate(bilinear_argmax, cases)
results_table(results).round(3)

In [ ]:
biou_per_class = pd.DataFrame({name: results[name]["biou_per_class"] for name in VAL_SETS}).T
biou_per_class.round(3)

In [ ]:
tile_image = next(a for a in val_sets["bushcha"] if a.date == LATENCY_TILE_DATE)
tile = standard_tile(tile_image.rgb, *LATENCY_TILE_TOP_LEFT_PX)
latency = model_latency(model, tile)
assert sum(p.numel() for p in model.parameters()) == n_params_initial
latency

# Plots

In [ ]:
fig_loss, ax = plt.subplots(figsize=(7, 3.5))
if history:
    frame = pd.DataFrame(history)
    ax.plot(frame["step"], frame["ce_dice"], "-o", ms=3, label="CE + Dice")
    ax.plot(frame["step"], frame["band_ce"], "-o", ms=3, label=f"band CE ({BAND_PX} px)")
ax.set_title(f"training loss, lambda = {LAMBDA_BOUNDARY:g}, seed {SEED}")
ax.set_xlabel("step"); ax.grid(alpha=.25); ax.legend()
fig_loss.tight_layout()

# Log

In [ ]:
mean = results[MEAN_KEY]
metrics = {
    "boundary_iou_mean": mean["biou"], "miou_mean": mean["miou"],
    "bf1_1px_mean": mean["bf1_1px"], "bf1_2px_mean": mean["bf1_2px"],
    "model_cpu_ms": latency.median_ms, "within_latency_budget": latency.median_ms <= LATENCY_BUDGET_MS,
    "train_minutes": train_minutes,
    **{f"boundary_iou_{name}": results[name]["biou"] for name in VAL_SETS},
    **{f"miou_{name}": results[name]["miou"] for name in VAL_SETS},
    "biou_per_class": {name: results[name]["biou_per_class"] for name in VAL_SETS},
    "iou_per_class": {name: results[name]["iou_per_class"] for name in VAL_SETS},
    "bf1_2px_per_class": {name: results[name]["bf1_2px_per_class"] for name in VAL_SETS},
    "loss_history": history,
}
metrics["boundary_iou_mean"], metrics["miou_mean"], metrics["model_cpu_ms"]

In [ ]:
RUN_CONFIG = {"dataset": dataset_ref(DATASET), "init_checkpoint": INIT_CHECKPOINT, "lambda_boundary": LAMBDA_BOUNDARY,
              "band_px": BAND_PX, "steps": STEPS, "lr": LR, "weight_decay": WEIGHT_DECAY, "warmup_frac": WARMUP_FRAC,
              "crop_px": SAT_CROP_PX, "batch": SAT_BATCH, "rare_crop_p": RARE_CROP_P, "gap_px": GAP_PX,
              "recalibrate_bn": RECALIBRATE_BN, "bn_batches": BN_BATCHES, "bn_seed": BN_SEED,
              "train_threads": TRAIN_THREADS, "eval_device": EVAL_DEVICE}
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_loss], seed=SEED,
              checkpoint=str(checkpoint.relative_to(REPO_ROOT)) if checkpoint else None,
              device=DEVICE) if LOG_RUN else {"run": "dry"}
run["run"]